# 07 - Transformer Model (DistilBERT)

**Goal:** experiment with a modern transformer model for sentiment classification, then fairly compare it to the classical TF-IDF + Logistic Regression baseline.

**Model:** `distilbert-base-uncased` — a lightweight distilled version of BERT that keeps ~97% of BERT's language understanding at ~40% the size.

> **Requirements:** this notebook needs `transformers`, `torch`, and `datasets`. Install with:
> ```
> pip install transformers torch datasets
> ```
>
> Training DistilBERT takes a few minutes on CPU and downloads ~250 MB of weights. If hardware is limited, reduce `epochs` / `batch_size` or use a smaller sample.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.data_loader import load_raw_tweets

In [ ]:
# Use a manageable random sample so training finishes on CPU
df = pd.DataFrame({
    "feedback": load_raw_tweets()["text"].astype(str),
    "sentiment": load_raw_tweets()["airline_sentiment"].str.lower(),
}).sample(4000, random_state=42)
print(df["sentiment"].value_counts())

In [ ]:
from src.transformer_model import TransformerSentimentClassifier, SENTIMENT_MAP

clf = TransformerSentimentClassifier(num_labels=3)
train_ds, test_ds = clf.prepare_data(
    df["feedback"].tolist(),
    df["sentiment"].tolist(),
    SENTIMENT_MAP,
)

In [ ]:
# Fine-tune on the sample (CPU: expect a few minutes for 2 - 3 epochs)
train_metrics = clf.train(
    train_ds,
    output_dir="./models/transformer_sentiment",
    epochs=2,
    batch_size=8,
)
print("Train metrics:", {k: round(v, 4) for k, v in train_metrics.items()})

In [ ]:
# Evaluate on the held-out test set
eval_metrics = clf.evaluate(test_ds)
print({k: round(v, 4) for k, v in eval_metrics.items()})

## Honest comparison: TF-IDF + LR vs DistilBERT

For a fair comparison we train the classical model on the **same** data split.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from src.sentiment import prepare_sentiment_data, train_sentiment_model

X_train, X_test, y_train, y_test, _ = prepare_sentiment_data(
    df, text_column="feedback", label_column="sentiment"
)
classical = train_sentiment_model(X_train, y_train)
y_pred = classical.predict(X_test)

classical_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}
print("Classical:", {k: round(v, 4) for k, v in classical_metrics.items()})

In [ ]:
# DistilBERT eval metrics from the built-in trainer
transformer_metrics = {
    "accuracy": eval_metrics.get("eval_accuracy", float("nan")),
    "precision": float("nan"),  # not directly reported by the trainer
    "recall": float("nan"),
    "f1": eval_metrics.get("eval_f1", float("nan")),
}

from src.transformer_model import compare_classical_vs_transformer

compare_classical_vs_transformer(classical_metrics, transformer_metrics)

## How to read the comparison

- If DistilBERT's F1 is clearly higher, the task benefits from contextual understanding (sarcasm, negation, word order).
- If they are close, the simpler and cheap TF-IDF + LR model is often a *better engineering choice*.
- DistilBERT always costs more: slower training, slower inference, more memory, and it is a black box.

**We do not claim the transformer is better unless the numbers show it.**

## Context understanding

- TF-IDF treats text as a **bag of words** — word order is lost, so "not good" and "good not" look identical.
- Transformers use **attention**, so they consider each word in the context of its neighbors. This is why they handle negation and longer-range dependencies better.